# 3. Video frames

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hpa-code/simcitynet/blob/main/docs/tutorials/03_video_frames.ipynb)

*Run it in Colab (free GPU) to see every figure; the website shows the saved outputs.*

The paper embeds every frame of clips generated by Sora and Kling. `embed_video` samples a
video at a fixed interval and embeds each frame. To keep this notebook self-contained we make
a short "camera pan" clip from a photo; replace `clip_path` with your own AI-generated or
recorded video.

In [ ]:
# Installs simcitynet with all models (PyTorch, Transformers, Ultralytics YOLO).
# In Colab, choose Runtime > Change runtime type > GPU for speed.
# Uses PyPI; until the first PyPI release it installs from GitHub instead.
!pip install -q "simcitynet[all]" || pip install -q "simcitynet[all] @ git+https://github.com/hpa-code/simcitynet.git"

In [ ]:
import cv2
import numpy as np
from PIL import Image
from simcitynet import SimCityNet
from simcitynet.datasets import fetch_examples

paths = fetch_examples("examples")
photo = np.asarray(Image.open("examples/zurich.jpg").convert("RGB"))

## Make a 6-second panning clip (skip this if you have a video)

In [ ]:
h, w = photo.shape[:2]
win_w = int(w * 0.6)
fps, seconds = 10, 6
clip_path = "pan.mp4"
writer = cv2.VideoWriter(clip_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (win_w, h))
for i in range(fps * seconds):
    x = int((w - win_w) * i / (fps * seconds - 1))
    writer.write(cv2.cvtColor(photo[:, x : x + win_w], cv2.COLOR_RGB2BGR))
writer.release()

## Embed one frame every half second

In [ ]:
scn = SimCityNet()
frames = scn.embed_video(clip_path, every=0.5)
frames.raw[["vegetation", "person", "car", "road_share", "building_share"]].round(3)

## How the scene changes over time

In [ ]:
import matplotlib.pyplot as plt

t = frames.meta["timestamp"]
s = frames.summary()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=True)
for ax, col, label in zip(axes, ["greenery", "building", "pedestrians"],
                          ["Greenery (depth-weighted)", "Building share", "People counted"]):
    ax.plot(t, s[col], color="#2a78d6", linewidth=2, marker="o", markersize=4)
    ax.set_title(label, loc="left", fontsize=10)
    ax.set_xlabel("seconds")
    ax.grid(color="#e4e3df")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.tight_layout()

In [ ]:
frames.plot.scene_card(0)

## Which real place does the clip resemble?

Combine the frames with the six photos and look at each frame's most similar photo.

In [ ]:
import pandas as pd
from simcitynet import Embeddings

photos = scn.embed(paths)
combined = Embeddings(
    pd.concat([photos.raw, frames.raw]),
    scenes=[*photos.scenes, *frames.scenes],
)
sim = combined.similarity()
best = sim.loc[frames.names, photos.names].idxmax(axis=1)
best.value_counts()

In [ ]:
combined.plot.map()

## With your own clips

```python
clip = scn.embed_video("sora_doha_souq.mp4", every=1.0)
kling = scn.embed_video("kling_doha_souq.mp4", every=1.0)
```

Then compare the frame distributions of each model with real photos of the same place — the
core design of the paper.